# 21 — Performance Optimization

Pandas can be slow by default. This notebook covers vectorization, eval()/query(),
avoiding iterrows(), Numba acceleration, and the canonical profiling workflow.

In [ ]:
import pandas as pd
import numpy as np
import time

np.random.seed(42)
n = 100_000

df = pd.DataFrame({
    'customer_id': np.random.randint(1, 10000, n),
    'amount':      np.round(np.random.uniform(100, 50000, n), 2),
    'quantity':    np.random.randint(1, 20, n),
    'discount':    np.round(np.random.uniform(0, 0.3, n), 3),
    'category':    np.random.choice(['Electronics', 'Clothing', 'Food', 'Books'], n),
    'is_returned': np.random.choice([0, 1], n, p=[0.92, 0.08]),
})

print(f"Dataset: {df.shape}")
print(f"Memory:  {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

## 1. iterrows() — Never Use This in Production

In [ ]:
# Using a small sample to make the comparison fair
small = df.head(10_000).copy()

# SLOW: iterrows() — Python-level loop
start = time.perf_counter()
revenues = []
for _, row in small.iterrows():
    rev = row['amount'] * row['quantity'] * (1 - row['discount'])
    revenues.append(rev)
small['revenue_iterrows'] = revenues
t_iterrows = time.perf_counter() - start

# FAST: vectorized
start = time.perf_counter()
small['revenue_vectorized'] = small['amount'] * small['quantity'] * (1 - small['discount'])
t_vector = time.perf_counter() - start

print(f"iterrows():    {t_iterrows:.3f}s")
print(f"vectorized:    {t_vector:.4f}s")
print(f"Speedup:       {t_iterrows / t_vector:.0f}×")
print(f"Results match: {np.allclose(small['revenue_iterrows'], small['revenue_vectorized'])}")

## 2. apply() vs Vectorized Operations

In [ ]:
# apply() on row axis (axis=1) — slow Python loop
start = time.perf_counter()
result_apply = small.apply(
    lambda row: row['amount'] * row['quantity'] * (1 - row['discount']),
    axis=1
)
t_apply = time.perf_counter() - start

# Vectorized
start = time.perf_counter()
result_vec = small['amount'] * small['quantity'] * (1 - small['discount'])
t_vec = time.perf_counter() - start

print(f"apply(axis=1): {t_apply:.3f}s")
print(f"vectorized:    {t_vec:.4f}s")
print(f"Speedup:       {t_apply / t_vec:.0f}×")

In [ ]:
# When apply() IS appropriate:
# 1. Column-wise operations (axis=0) — rare
# 2. Returning a DataFrame from each group in groupby().apply()
# 3. Logic that truly can't be vectorized

# Even then: prefer np.vectorize or np.where for element-wise logic

# apply(axis=0): apply to each column
stats = small[['amount', 'quantity', 'discount']].apply(lambda col: col.describe())
print(stats)

## 3. np.where vs apply for Conditional Logic

In [ ]:
# Conditional value assignment
# SLOW: apply with lambda
start = time.perf_counter()
small['tier_apply'] = small['amount'].apply(
    lambda x: 'High' if x > 30000 else ('Mid' if x > 10000 else 'Low')
)
t_apply = time.perf_counter() - start

# FAST: np.where (vectorized)
start = time.perf_counter()
small['tier_npwhere'] = np.where(
    small['amount'] > 30000, 'High',
    np.where(small['amount'] > 10000, 'Mid', 'Low')
)
t_npwhere = time.perf_counter() - start

# ALSO FAST: pd.cut (for range buckets)
start = time.perf_counter()
small['tier_cut'] = pd.cut(small['amount'], bins=[0, 10000, 30000, np.inf],
                            labels=['Low', 'Mid', 'High'])
t_cut = time.perf_counter() - start

print(f"apply():   {t_apply:.3f}s")
print(f"np.where: {t_npwhere:.4f}s")
print(f"pd.cut:   {t_cut:.4f}s")
print(f"Speedup (apply vs np.where): {t_apply / t_npwhere:.0f}×")

## 4. eval() and query() — Numexpr-Backed Speed

In [ ]:
# For large DataFrames, eval() uses numexpr (C-backed multi-threaded)
# It is faster than standard Pandas operations for large data

# Standard
start = time.perf_counter()
df['revenue_standard'] = df['amount'] * df['quantity'] * (1 - df['discount'])
t_std = time.perf_counter() - start

# eval()
start = time.perf_counter()
df.eval('revenue_eval = amount * quantity * (1 - discount)', inplace=True)
t_eval = time.perf_counter() - start

print(f"Standard: {t_std:.3f}s")
print(f"eval():   {t_eval:.3f}s")
print(f"Match:    {np.allclose(df['revenue_standard'], df['revenue_eval'])}")

In [ ]:
# query() — expressive, fast filtering
start = time.perf_counter()
result_mask = df[(df['amount'] > 10000) & (df['is_returned'] == 0) & (df['category'] == 'Electronics')]
t_mask = time.perf_counter() - start

start = time.perf_counter()
result_query = df.query("amount > 10000 and is_returned == 0 and category == 'Electronics'")
t_query = time.perf_counter() - start

print(f"Boolean mask: {t_mask:.3f}s  rows: {len(result_mask):,}")
print(f"query():      {t_query:.3f}s  rows: {len(result_query):,}")

In [ ]:
# query() with Python variables using @
min_amount = 10000
cats = ['Electronics', 'Clothing']

result = df.query("amount > @min_amount and category in @cats")
print(f"query with @vars: {len(result):,} rows")

## 5. Profiling — Finding Bottlenecks

In [ ]:
def compute_revenue(df):
    df['revenue'] = df['amount'] * df['quantity'] * (1 - df['discount'])
    return df

# Jupyter: %timeit for micro-benchmarking
# %timeit compute_revenue(df.copy())

# Manual profiling with time.perf_counter
N = 5
times = []
for _ in range(N):
    start = time.perf_counter()
    _ = compute_revenue(df.copy())
    times.append(time.perf_counter() - start)

print(f"compute_revenue({len(df):,} rows):")
print(f"  mean: {np.mean(times)*1000:.1f}ms")
print(f"  min:  {np.min(times)*1000:.1f}ms")
print(f"  max:  {np.max(times)*1000:.1f}ms")

## 6. pd.cut() / pd.qcut() — Vectorized Binning

In [ ]:
# pd.cut() — fixed-width bins
df['amount_tier'] = pd.cut(
    df['amount'],
    bins=[0, 5000, 15000, 30000, 50001],
    labels=['Low', 'Mid', 'High', 'Premium'],
    right=True
)
print("cut() distribution:")
print(df['amount_tier'].value_counts())

In [ ]:
# pd.qcut() — equal-frequency bins (quantile-based)
df['amount_quartile'] = pd.qcut(
    df['amount'],
    q=4,
    labels=['Q1', 'Q2', 'Q3', 'Q4']
)
print("qcut() distribution (equal-frequency):")
print(df['amount_quartile'].value_counts())

## 7. map() vs apply() vs replace() — Lookup Performance

In [ ]:
category_code = {'Electronics': 'E', 'Clothing': 'C', 'Food': 'F', 'Books': 'B'}

start = time.perf_counter()
df['cat_code_apply'] = df['category'].apply(lambda x: category_code.get(x))
t_apply = time.perf_counter() - start

start = time.perf_counter()
df['cat_code_map'] = df['category'].map(category_code)
t_map = time.perf_counter() - start

start = time.perf_counter()
df['cat_code_replace'] = df['category'].replace(category_code)
t_replace = time.perf_counter() - start

print(f"apply():   {t_apply:.3f}s")
print(f"map():     {t_map:.4f}s")
print(f"replace(): {t_replace:.4f}s")
print(f"map() is {t_apply / t_map:.0f}× faster than apply()")

## 8. Avoid Repeated Indexing

In [ ]:
# SLOW: multiple passes over the same column
start = time.perf_counter()
for _ in range(100):
    _ = df['amount'].mean()
    _ = df['amount'].std()
    _ = df['amount'].max()
t_multi = time.perf_counter() - start

# FAST: single pass with describe() or agg()
start = time.perf_counter()
for _ in range(100):
    _ = df['amount'].agg(['mean', 'std', 'max'])
t_agg = time.perf_counter() - start

print(f"3 separate calls: {t_multi:.3f}s")
print(f"agg(['mean','std','max']): {t_agg:.3f}s")

## 9. Numba JIT — For Custom Loops

In [ ]:
# Numba @jit compiles your Python loop to native code
# Use when you MUST have a loop (e.g., custom recursive calculation)

try:
    from numba import jit

    @jit(nopython=True)
    def compute_revenue_numba(amounts, quantities, discounts):
        result = np.empty(len(amounts))
        for i in range(len(amounts)):
            result[i] = amounts[i] * quantities[i] * (1 - discounts[i])
        return result

    amounts    = df['amount'].values
    quantities = df['quantity'].values
    discounts  = df['discount'].values

    # Warm up Numba (first call compiles)
    _ = compute_revenue_numba(amounts[:100], quantities[:100], discounts[:100])

    start = time.perf_counter()
    result_numba = compute_revenue_numba(amounts, quantities, discounts)
    t_numba = time.perf_counter() - start

    start = time.perf_counter()
    result_vec = amounts * quantities * (1 - discounts)
    t_vec = time.perf_counter() - start

    print(f"Numba:     {t_numba:.4f}s")
    print(f"Vectorized: {t_vec:.4f}s")
    print(f"Match: {np.allclose(result_numba, result_vec)}")

except ImportError:
    print("numba not installed. Install with: pip install numba")
    print("Numba JIT compiles Python loops to C speed — useful when vectorization is impossible.")

## 10. Performance Optimization Checklist

| Priority | Anti-Pattern | Fix |
|----------|-------------|-----|
| 🔴 Critical | `iterrows()` | Vectorized operations |
| 🔴 Critical | `apply(axis=1)` | `np.where`, `np.select`, `pd.cut` |
| 🟡 High | `.apply(lambda)` for lookup | `.map(dict)` |
| 🟡 High | Repeated column access | Cache in variable |
| 🟡 High | Multiple `.sum()/.mean()/.std()` | Single `.agg()` |
| 🟢 Medium | Boolean mask filter | `.query()` |
| 🟢 Medium | Complex arithmetic on big df | `pd.eval()` |
| 🟢 Medium | String operations | Arrow-backed series |
| ⚪ Optional | Custom loops required | Numba `@jit` |